# Urban Sewage Metagenome Sample Metadata and Antibiotic Resistance Gene Tables from 101 Countries Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

This notebook follows a step-by-step workflow:
- load the Croissant metadata,
- inspect available record sets using their `@id` values,
- load records into pandas DataFrames,
- perform exploratory data analysis with dynamic variable selection,
- and create simple visualizations.

> Note: all references to record sets, fields, and columns in this notebook use their Croissant `@id` values whenever available.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata from the dataset using `mlcroissant`.

We keep the dataset URL in a variable so the notebook is easy to reuse with another Croissant schema.

In [ ]:
import json
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.abtv-wmj4/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset loaded successfully.")
print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}")

In [ ]:
# Convert metadata to JSON for easier inspection when needed
metadata_json = metadata.to_json()

summary = {
    "@id": metadata_json.get("@id"),
    "name": metadata_json.get("name"),
    "identifier": metadata_json.get("identifier"),
    "version": metadata_json.get("version"),
    "license": metadata_json.get("license"),
    "spatialCoverage": metadata_json.get("spatialCoverage"),
    "temporalCoverage": metadata_json.get("temporalCoverage"),
}

summary

## 2. Data Overview
Review available record sets, fields, and their IDs.

Some Croissant datasets expose multiple record sets. This dataset metadata indicates that `recordSet` may be empty, so the following code safely inspects what is available and adapts automatically.

The goal here is to identify every data entity by `@id` and create variables that can be reused later.

In [ ]:
# Inspect top-level metadata keys
sorted(metadata_json.keys())

In [ ]:
# Extract record set information from the metadata JSON
record_sets_json = metadata_json.get("recordSet", []) or []

print(f"Number of record sets declared in metadata: {len(record_sets_json)}")

record_set_rows = []
for rs in record_sets_json:
    rs_id = rs.get("@id")
    rs_name = rs.get("name")
    rs_description = rs.get("description")
    rs_fields = rs.get("field", []) or []
    record_set_rows.append({
        "record_set_id": rs_id,
        "record_set_name": rs_name,
        "field_count": len(rs_fields),
        "description": rs_description,
    })

record_sets_df = pd.DataFrame(record_set_rows)
record_sets_df

In [ ]:
# Build a field-level catalog using @id references
field_rows = []
for rs in record_sets_json:
    rs_id = rs.get("@id")
    rs_name = rs.get("name")
    for field in rs.get("field", []) or []:
        source = field.get("source", {}) or {}
        extract = source.get("extract", {}) if isinstance(source, dict) else {}
        field_rows.append({
            "record_set_id": rs_id,
            "record_set_name": rs_name,
            "field_id": field.get("@id"),
            "field_name": field.get("name"),
            "data_type": field.get("dataType"),
            "description": field.get("description"),
            "source_file": source.get("fileObject") if isinstance(source, dict) else None,
            "source_field": source.get("field") if isinstance(source, dict) else None,
            "extract_column": extract.get("column") if isinstance(extract, dict) else None,
        })

fields_df = pd.DataFrame(field_rows)
fields_df.head(20)

If record sets are available, the next cell prints a few sample records for each `record_set_id`. Otherwise, it explains that no record sets were declared in the supplied metadata.

In [ ]:
record_set_ids = [rs.get("@id") for rs in record_sets_json if rs.get("@id")]
print("Record set @ids:", record_set_ids)

if not record_set_ids:
    print("No record sets were declared in the Croissant metadata. Data record loading will be skipped unless record sets are added in a future version.")
else:
    for record_set_id in record_set_ids:
        print(f"\nPreviewing records from record set: {record_set_id}")
        try:
            for i, record in enumerate(dataset.records(record_set=record_set_id)):
                print(record)
                if i >= 2:
                    break
        except Exception as e:
            print(f"Could not load records for {record_set_id}: {e}")

## 3. Data Extraction
Load data from each available record set into a DataFrame for analysis.

This section uses a dictionary keyed by the record set `@id`, which makes the notebook robust and consistent with Croissant identifiers.

In [ ]:
# Extract data from each record set into a pandas DataFrame
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from {record_set_id}")
    except Exception as e:
        print(f"Skipping {record_set_id} because records could not be loaded: {e}")

print("\nAvailable DataFrames:")
print(list(dataframes.keys()))

In [ ]:
# Inspect the first available DataFrame, if any
selected_record_set_id = next(iter(dataframes), None)

if selected_record_set_id is None:
    print("No DataFrames were created because no loadable record sets are available in the metadata.")
else:
    print(f"Selected record set: {selected_record_set_id}")
    print("Columns:")
    print(dataframes[selected_record_set_id].columns.tolist())
    display(dataframes[selected_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as:
- selecting numeric fields dynamically,
- filtering rows based on a threshold,
- normalizing a numeric field,
- and grouping by a categorical field.

The code below is defensive: it checks whether record sets and appropriate columns are actually available before running each step.

In [ ]:
if selected_record_set_id is None:
    print("EDA skipped because no record set data is available.")
else:
    analysis_df = dataframes[selected_record_set_id].copy()
    numeric_columns = analysis_df.select_dtypes(include="number").columns.tolist()
    object_columns = analysis_df.select_dtypes(include=["object", "category"]).columns.tolist()

    print(f"Numeric columns in {selected_record_set_id}: {numeric_columns}")
    print(f"Categorical/text columns in {selected_record_set_id}: {object_columns[:20]}")

In [ ]:
if selected_record_set_id is None:
    print("No analysis performed.")
else:
    analysis_df = dataframes[selected_record_set_id].copy()
    numeric_columns = analysis_df.select_dtypes(include="number").columns.tolist()
    categorical_columns = analysis_df.select_dtypes(include=["object", "category"]).columns.tolist()

    if not numeric_columns:
        print("No numeric fields found in the selected record set, so numeric filtering and normalization are skipped.")
    else:
        numeric_field_id = numeric_columns[0]
        threshold = analysis_df[numeric_field_id].median()
        filtered_df = analysis_df[analysis_df[numeric_field_id] > threshold].copy()

        print(f"Using numeric field: {numeric_field_id}")
        print(f"Threshold (median): {threshold}")
        print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} rows")
        display(filtered_df.head())

        std_value = filtered_df[numeric_field_id].std()
        if std_value is not None and pd.notna(std_value) and std_value != 0:
            normalized_field_id = f"{numeric_field_id}_normalized"
            filtered_df[normalized_field_id] = (
                (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / std_value
            )
            print(f"Normalized field created: {normalized_field_id}")
            display(filtered_df[[numeric_field_id, normalized_field_id]].head())
        else:
            print("Normalization skipped because the standard deviation is zero or undefined.")

        if categorical_columns:
            group_field_id = categorical_columns[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean(numeric_only=True).sort_values(ascending=False)
            print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head(10).to_frame(name=f"mean_{numeric_field_id}"))
        else:
            print("Grouping skipped because no categorical field is available.")

### Metadata-only exploration
Because the supplied Croissant metadata may not expose record sets directly, it is still useful to analyze dataset-level properties such as keywords, citations, distributions, and narrative documentation.

In [ ]:
metadata_exploration = {
    "keyword_count": len(metadata_json.get("keywords", []) or []),
    "citation_count": len(metadata_json.get("citation", []) or []),
    "distribution_count": len(metadata_json.get("distribution", []) or []),
    "author_count": len(metadata_json.get("author", []) or []),
    "data_bias_count": len(metadata_json.get("dataBiases", []) or []),
    "data_limitations_count": len(metadata_json.get("dataLimitations", []) or []),
    "data_use_cases_count": len(metadata_json.get("dataUseCases", []) or []),
}

pd.DataFrame([metadata_exploration])

## 5. Visualization
Visualize either record-level data, if available, or dataset-level metadata summaries otherwise.

In [ ]:
if selected_record_set_id is not None:
    viz_df = dataframes[selected_record_set_id].copy()
    numeric_columns = viz_df.select_dtypes(include="number").columns.tolist()

    if numeric_columns:
        numeric_field_id = numeric_columns[0]
        plt.figure(figsize=(10, 5))
        sns.histplot(viz_df[numeric_field_id].dropna(), bins=30, kde=True)
        plt.title(f"Distribution of {numeric_field_id} in {selected_record_set_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.tight_layout()
        plt.show()
    else:
        print("No numeric field available for record-level visualization.")
else:
    counts_series = pd.Series(metadata_exploration).sort_values(ascending=False)
    plt.figure(figsize=(10, 5))
    sns.barplot(x=counts_series.index, y=counts_series.values, palette="viridis")
    plt.title("Dataset-level metadata summary counts")
    plt.xlabel("Metadata component")
    plt.ylabel("Count")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
# Visualize keyword frequencies by simple listing length and inspect the keyword set
keywords = metadata_json.get("keywords", []) or []
keywords_df = pd.DataFrame({"keyword": keywords})
display(keywords_df)

if not keywords_df.empty:
    plt.figure(figsize=(12, max(4, len(keywords_df) * 0.35)))
    sns.barplot(data=keywords_df.assign(count=1), y="keyword", x="count", color="steelblue")
    plt.title("Dataset keywords")
    plt.xlabel("Presence")
    plt.ylabel("Keyword")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrated how to:
- load a Croissant dataset with `mlcroissant`,
- inspect dataset metadata through a single metadata object,
- enumerate record sets and fields by their `@id` values,
- load records into pandas DataFrames when available,
- and perform basic EDA and visualization.

For this dataset, the supplied metadata is especially rich at the dataset level, including extensive provenance, bias statements, use cases, and documentation about synthetic versus directly sourced data. If future versions expose populated `recordSet` entries, the same notebook structure can be reused immediately because it already handles record-set IDs dynamically.